In [2]:
import os
import subprocess

def run_shell_command_capture(command, cwd=None, env=None):
    """Executes a shell command, captures and prints its output and error, and returns its exit code."""
    print(f"Executing: {command}")
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True, check=False, cwd=cwd, env=env)
        if result.stdout:
            print("--- STDOUT ---")
            print(result.stdout)
        if result.stderr:
            print("--- STDERR ---")
            print(result.stderr)
        print(f"Exit Code: {result.returncode}")
        return result.returncode
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return -1

# --- GLIBC 2.39 Build Process ---
print("--- Starting GLIBC 2.39 Build Process ---")

# 1. Download and extract GLIBC source
print("Downloading and extracting GLIBC 2.39...")
!wget http://ftp.gnu.org/gnu/libc/glibc-2.39.tar.gz
!tar -xvzf glibc-2.39.tar.gz

# 2. Install dependencies
print("Installing build dependencies...")
!apt-get update
!apt-get install -y gcc make gdb texinfo gawk bison sed python3-dev python3-pip python-is-python3
!pip install pexpect

# Change to glibc source directory
%cd /content/glibc-2.39

# Re-create build directory and navigate into it
!rm -rf build
!mkdir build
%cd build

# 3. Configure the build with a user-writable prefix
print("Re-configuring GLIBC build with local prefix...")
# New prefix will be /content/glibc-2.39/local_glibc
!../configure --prefix=/content/glibc-2.39/local_glibc \
  --host=x86_64-linux-gnu \
  --build=x86_64-linux-gnu \
  CC="gcc -m64" \
  CXX="g++ -m64" \
  CFLAGS="-O2" \
  CXXFLAGS="-O2"

print("\n--- Verifying 'prefix' in config.make immediately after configure ---")
!grep "^prefix" config.make || echo "'prefix' not found in config.make or configure failed to set it."

print("\n--- Displaying full config.make for inspection (first 20 lines) ---")
!head -n 20 config.make

# Compile GLIBC
print("Re-compiling GLIBC with full verbose output to a log file...")
!make clean > make_clean_log.txt 2>&1
!make -j$(nproc) V=1 &> make_compile_log.txt
!echo $? > make_exit_status.txt

# 5. Install GLIBC locally to the specified prefix
print("Installing GLIBC to /content/glibc-2.39/local_glibc...")
%env LANG=C
%env LC_ALL=C
!make install

# --- Reinstalling `mesh-llm` ---
print("\n--- Reinstalling mesh-llm after Runtime Restart ---")
!curl -fsSL https://raw.githubusercontent.com/Mesh-LLM/mesh-llm/main/install.sh | bash

# Verify mesh-llm installation
print("\nVerifying mesh-llm executable presence:")
!ls -l /root/.local/bin/mesh-llm
!file /root/.local/bin/mesh-llm

# --- Re-verifying GLIBC Installation and `mesh-llm` Linkage ---
print("\n--- Re-verifying GLIBC installation directories ---")
custom_glibc_path = '/content/glibc-2.39/local_glibc/lib'
mesh_llm_executable = '/root/.local/bin/mesh-llm'

print(f"Contents of /content/glibc-2.39/local_glibc/lib:")
run_shell_command_capture(f"ls -l {custom_glibc_path}")

print(f"\nContents of /content/glibc-2.39/local_glibc/include:")
run_shell_command_capture(f"ls -l /content/glibc-2.39/local_glibc/include")

env_vars = os.environ.copy()
env_vars['LD_LIBRARY_PATH'] = f"{custom_glibc_path}:/usr/local/lib:/usr/lib/x86_64-linux-gnu:/lib/x86_64-linux-gnu"
env_vars['PATH'] = f"/root/.local/bin:" + env_vars.get('PATH', '')

print(f"\n--- Re-inspecting {mesh_llm_executable} dependencies with ldd ---")
print(f"Using LD_LIBRARY_PATH={env_vars['LD_LIBRARY_PATH']}")
run_shell_command_capture(f"ldd {mesh_llm_executable}", env=env_vars)
print("\n--- End of ldd inspection ---")

# --- Attempting to run `mesh-llm` with the custom GLIBC dynamic linker ---
print("\n--- Attempting to run mesh-llm with custom dynamic linker ---")
custom_glibc_root = '/content/glibc-2.39/local_glibc'
custom_glibc_lib_path = f'{custom_glibc_root}/lib'
custom_dynamic_linker = f'{custom_glibc_lib_path}/ld-linux-x86-64.so.2'

print(f"Custom dynamic linker: {custom_dynamic_linker}")
print(f"Custom library path: {custom_glibc_lib_path}")

command_to_run = f"{custom_dynamic_linker} --library-path {custom_glibc_lib_path} {mesh_llm_executable} serve --auto"
run_shell_command_capture(command_to_run, env=env_vars)
print("\n--- End of mesh-llm execution attempt ---")

Streaming output truncated to the last 5000 lines.
glibc-2.39/sysdeps/x86_64/fpu/svml_s_log1pf8_core_avx.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_log2f16_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_log2f4_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_log2f8_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_log2f8_core_avx.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_logf16_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_logf4_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_logf8_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_logf8_core_avx.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_logf_data.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_logf_data.h
glibc-2.39/sysdeps/x86_64/fpu/svml_s_powf16_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_powf4_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_powf8_core.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_powf8_core_avx.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_powf_data.S
glibc-2.39/sysdeps/x86_64/fpu/svml_s_powf_data.h
glibc-2.39/sysdeps/x86_64/fpu/svml_s_sincosf16_core.S
glibc-2.39/sysdeps/x86_6

### Reinstalling `mesh-llm` and Re-evaluating GLIBC Dependency

In [5]:
print("Reinstalling mesh-llm...")
!curl -fsSL https://raw.githubusercontent.com/Mesh-LLM/mesh-llm/main/install.sh | bash

# Verify mesh-llm installation
print("\nVerifying mesh-llm executable presence:")
!ls -l /root/.local/bin/mesh-llm
!file /root/.local/bin/mesh-llm

Reinstalling mesh-llm...
Installing flavor: cpu
Release channel: stable
Installed mesh-llm-x86_64-unknown-linux-gnu.tar.gz to /root/.local/bin

/root/.local/bin is not on your PATH.
Add it with one of these commands:

bash:
  echo 'export PATH="/root/.local/bin:$PATH"' >> ~/.bashrc
  source ~/.bashrc

zsh:
  echo 'export PATH="/root/.local/bin:$PATH"' >> ~/.zshrc
  source ~/.zshrc

Verifying mesh-llm executable presence:
-rwxr-xr-x 1 1001 1001 102905768 May  4 23:25 /root/.local/bin/mesh-llm
/root/.local/bin/mesh-llm: ELF 64-bit LSB pie executable, x86-64, version 1 (SYSV), dynamically linked, interpreter /lib64/ld-linux-x86-64.so.2, for GNU/Linux 3.2.0, BuildID[sha1]=1757e6bbf0a39f737f8e8a12d1c67ae3556c2f07, not stripped


In [6]:
print("\nChecking system's default GLIBC version:")
!ldd --version


Checking system's default GLIBC version:
ldd (Ubuntu GLIBC 2.35-0ubuntu3.13) 2.35
Copyright (C) 2022 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
Written by Roland McGrath and Ulrich Drepper.


In [7]:
print("\nChecking mesh-llm's GLIBC dependencies with system ldd:")
!ldd /root/.local/bin/mesh-llm


Checking mesh-llm's GLIBC dependencies with system ldd:
/root/.local/bin/mesh-llm: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.38' not found (required by /root/.local/bin/mesh-llm)
/root/.local/bin/mesh-llm: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.39' not found (required by /root/.local/bin/mesh-llm)
	linux-vdso.so.1 (0x00007fff4d7ba000)
	libssl.so.3 => /lib/x86_64-linux-gnu/libssl.so.3 (0x00007f8f7815c000)
	libcrypto.so.3 => /lib/x86_64-linux-gnu/libcrypto.so.3 (0x00007f8f77d18000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x00007f8f77cf8000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x00007f8f77c11000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007f8f779e8000)
	/lib64/ld-linux-x86-64.so.2 (0x00007f8f7d474000)


### New Strategy: Using `patchelf` with Pre-extracted GLIBC 2.39

In [8]:
print("Installing patchelf...")
!apt-get update && apt-get install -y patchelf

Installing patchelf...
Get:1 https://cli.github.com/packages stable InRelease [3917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3917 B in 2s (2552 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages wil

In [14]:
print("\nDownloading GLIBC 2.39 .deb package from Ubuntu 24.04...")
# This specific URL might need updates if the package version changes or mirror changes.
# We will explicitly name the downloaded file.
deb_file_name = "libc6_2.39-0ubuntu8_amd64.deb"
!wget -O {deb_file_name} http://mirrors.kernel.org/ubuntu/pool/main/g/glibc/libc6_2.39-0ubuntu8_amd64.deb

# Install zstd for handling .zst archives
print("Installing zstd...")
!apt-get update -qq && apt-get install -y zstd

# Create a directory to extract GLIBC
glibc_extract_dir = "/content/glibc_2_39_extracted"
print(f"Creating extraction directory: {glibc_extract_dir}")
!rm -rf {glibc_extract_dir} # Clean up previous attempt
!mkdir -p {glibc_extract_dir}

# Extract the .deb package contents
print("Extracting GLIBC 2.39 .deb package using ar and tar...")

# Use 'ar -t' to list contents and find the data archive (usually data.tar.xz or data.tar.gz or data.tar.zst)
archive_contents = !ar -t {deb_file_name}
data_archive_name = next((s for s in archive_contents if s.startswith('data.tar.')), None)

if data_archive_name:
    print(f"Found data archive: {data_archive_name}. Extracting...")
    !ar x {deb_file_name} {data_archive_name}
    !tar -xf {data_archive_name} -C {glibc_extract_dir}
    # Clean up the extracted tarball as it's no longer needed
    !rm {data_archive_name}
else:
    print("Error: Could not find data archive (data.tar.xz, data.tar.gz, or data.tar.zst) in the .deb file.")

# Verify extracted contents to ensure libc.so.6 and ld-linux-x86-64.so.2 are present
print("\nVerifying extracted GLIBC contents:")
!ls -l {glibc_extract_dir}/lib/x86_64-linux-gnu/ || echo "Failed to list /lib/x86_64-linux-gnu (directory might not exist)"
!ls -l {glibc_extract_dir}/lib/x86_64-linux-gnu/libc.so.6 || echo "libc.so.6 not found"
!ls -l {glibc_extract_dir}/lib/x86_64-linux-gnu/ld-linux-x86-64.so.2 || echo "ld-linux-x86-64.so.2 not found"

# Also list the full contents of the extracted directory to confirm structure
print("\nFull contents of the extracted directory:")
!ls -RF {glibc_extract_dir}


--2026-05-18 21:30:32--  http://mirrors.kernel.org/ubuntu/pool/main/g/glibc/libc6_2.39-0ubuntu8_amd64.deb
Resolving mirrors.kernel.org (mirrors.kernel.org)... 142.0.200.124, 2605:f480:58:1:0:1994:3:14
Connecting to mirrors.kernel.org (mirrors.kernel.org)|142.0.200.124|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://mirrors.edge.kernel.org/ubuntu/pool/main/g/glibc/libc6_2.39-0ubuntu8_amd64.deb [following]
--2026-05-18 21:30:33--  http://mirrors.edge.kernel.org/ubuntu/pool/main/g/glibc/libc6_2.39-0ubuntu8_amd64.deb
Resolving mirrors.edge.kernel.org (mirrors.edge.kernel.org)... 213.196.21.55, 2a01:60a::1994:3:14
Connecting to mirrors.edge.kernel.org (mirrors.edge.kernel.org)|213.196.21.55|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3264806 (3.1M) [application/octet-stream]
Saving to: 'libc6_2.39-0ubuntu8_amd64.deb'

libc6_2.39-0ubuntu8 100%[===================>]   3.11M  2.74MB/s    in 1.1s    

2026-05-18 21:3

In [15]:
mesh_llm_executable = '/root/.local/bin/mesh-llm'

# Corrected paths based on extraction output
new_interpreter = f"{glibc_extract_dir}/usr/lib64/ld-linux-x86-64.so.2"
new_rpath = f"{glibc_extract_dir}/usr/lib/x86_64-linux-gnu"

print(f"\nUsing patchelf to modify {mesh_llm_executable}...")
print(f"Setting new interpreter: {new_interpreter}")
print(f"Adding new RPATH: {new_rpath}")

# Backup the original executable before modification
!cp {mesh_llm_executable} {mesh_llm_executable}.bak

# Modify the executable to use the new interpreter and RPATH
# We check if the existing RPATH is empty to avoid adding a leading colon.
# If it's not empty, we append the new RPATH, otherwise we just set the new RPATH.
existing_rpath_command = f"patchelf --print-rpath {mesh_llm_executable}"
result = !{existing_rpath_command}
existing_rpath = result[0].strip()

if existing_rpath:
    !patchelf --set-rpath {new_rpath}:{existing_rpath} {mesh_llm_executable}
else:
    !patchelf --set-rpath {new_rpath} {mesh_llm_executable}

!patchelf --set-interpreter {new_interpreter} {mesh_llm_executable}


Using patchelf to modify /root/.local/bin/mesh-llm...
Setting new interpreter: /content/glibc_2_39_extracted/usr/lib64/ld-linux-x86-64.so.2
Adding new RPATH: /content/glibc_2_39_extracted/usr/lib/x86_64-linux-gnu


In [16]:
mesh_llm_executable = '/root/.local/bin/mesh-llm'

# Re-check mesh-llm's GLIBC dependencies with the modified binary
print(f"\nRe-checking {mesh_llm_executable} dependencies after patchelf:")
!ldd {mesh_llm_executable}

# Attempt to run mesh-llm
print("\nAttempting to run mesh-llm after patchelf modification:")
!{mesh_llm_executable} serve --auto


Re-checking /root/.local/bin/mesh-llm dependencies after patchelf:
	linux-vdso.so.1 (0x00007fff8af80000)
	libssl.so.3 => /lib/x86_64-linux-gnu/libssl.so.3 (0x0000794bf010d000)
	libcrypto.so.3 => /lib/x86_64-linux-gnu/libcrypto.so.3 (0x0000794befcc9000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x0000794befca9000)
	libm.so.6 => /content/glibc_2_39_extracted/usr/lib/x86_64-linux-gnu/libm.so.6 (0x0000794befbc0000)
	libc.so.6 => /content/glibc_2_39_extracted/usr/lib/x86_64-linux-gnu/libc.so.6 (0x0000794bef9ae000)
	/content/glibc_2_39_extracted/usr/lib64/ld-linux-x86-64.so.2 => /lib64/ld-linux-x86-64.so.2 (0x0000794bf63ed000)

Attempting to run mesh-llm after patchelf modification:
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
   🧾 Downloaded model metadata
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭
╭

## Getting Started with libp2p in Python

We'll install `py-libp2p` to explore basic `libp2p` concepts like peer IDs and simple communication.

In [46]:
import sys
import os

print("Changing to /content directory...")
%cd /content

print("Installing build-essential and libgmp-dev for C extensions...")
# Update apt-get and install build tools
!apt-get update -qq && apt-get install -y build-essential libgmp-dev

# Install direct Python dependencies first
print("Installing core Python dependencies (multiaddr, PyNaCl, fastecdsa, aioquic)...")
!{sys.executable} -m pip install multiaddr PyNaCl fastecdsa aioquic

print("Cloning py-libp2p repository...")
# Remove existing directory if it exists to ensure a clean clone
!rm -rf py-libp2p
!git clone https://github.com/libp2p/py-libp2p.git

# Change to the cloned directory
%cd /content/py-libp2p

print("Installing py-libp2p from source in editable mode...")
# This should now succeed with all build and runtime dependencies met.
# Use --no-deps to prevent pip from trying to reinstall already installed dependencies
!{sys.executable} -m pip install -e . --no-deps

print("py-libp2p and its dependencies installed successfully!")

Changing to /content directory...
/content
Installing build-essential and libgmp-dev for C extensions...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libgmp-dev is already the newest version (2:6.2.1+dfsg-3ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
Installing core Python dependencies (multiaddr, PyNaCl, fastecdsa, aioquic)...
  Using cached aioquic-1.3.0-cp310-abi3-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (6.3 kB)
  Using cached pylsqpack-0.3.24-cp310-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached service_identity-24.2.0-py3-none-any.whl.metadata (5.1 kB)
Using cached aioquic-1.3.0-cp3

Now, let's create a simple libp2p host and generate a Peer ID. A Peer ID uniquely identifies a node in the libp2p network.

In [43]:
from multiaddr import Multiaddr
from libp2p.peer.id import PeerID
from libp2p.crypto.rsa import create_new_key_pair

# 1. Demonstrate Multiaddresses
print("\n--- Libp2p Multiaddresses ---")
# A multiaddress represents a network address in a self-describing format
addr_tcp = Multiaddr("/ip4/127.0.0.1/tcp/4001")
addr_ws = Multiaddr("/ip4/192.168.1.100/tcp/8000/ws")

print(f"Example TCP Multiaddr: {addr_tcp}")
print(f"Example WebSocket Multiaddr: {addr_ws}")
print(f"Multiaddr bytes: {addr_tcp.to_bytes()}")
print(f"Multiaddr protocols: {[p.name for p in addr_tcp.protocols()]}")

# 2. Demonstrate Peer IDs
print("\n--- Libp2p Peer IDs ---")
# Generate a new RSA key pair
priv_key, pub_key = create_new_key_pair()

# Create a PeerID from the public key
peer_id = PeerID.from_pubkey(pub_key.serialize())

print(f"Generated Peer ID: {peer_id}")
print(f"Peer ID bytes: {peer_id.to_bytes().hex()}")
print(f"Peer ID (Qm... format): {peer_id.to_string()}")

# You can also create a PeerID from its string representation
some_peer_id_str = "QmYyQSo1c1Ym7orWxLYCrYp8F7w6ZrtfGBZRBFCFG8fRPi"
some_peer_id = PeerID.from_string(some_peer_id_str)
print(f"Peer ID from string '{some_peer_id_str}': {some_peer_id}")

KeyError: 'libp2p'

In [45]:
import sys
print("Python sys.path:")
for p in sys.path:
    print(p)

print("\nAttempting a basic import of libp2p...")
try:
    import libp2p
    print("Successfully imported libp2p module.")
except Exception as e:
    print(f"Error importing libp2p: {e}")

print("\nAttempting to import PeerID from libp2p.peer.id...")
try:
    from libp2p.peer.id import PeerID
    print("Successfully imported PeerID from libp2p.peer.id.")
except Exception as e:
    print(f"Error importing PeerID: {e}")


Python sys.path:
/content
/env/python
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython

Attempting a basic import of libp2p...
Error importing libp2p: No module named 'aioquic'

Attempting to import PeerID from libp2p.peer.id...
Error importing PeerID: 'libp2p'


This demonstrates creating a basic `libp2p` host and obtaining its unique Peer ID. To enable communication, the host would need to listen on a network address and have a way to discover other peers. We can explore that next, if you'd like!